# Softmax 函数学习

Softmax 常用在**多分类**任务的输出层：把一组任意实数（通常叫 *logits*）转换成一组**概率分布**。

你会在这里学到：
- softmax 的定义与直觉
- 数值稳定（减去最大值 / log-sum-exp）为什么必须做
- softmax 的梯度（Jacobian）以及和交叉熵的关系


## 1. 定义与性质

对向量 $z \in \mathbb{R}^C$，softmax 定义为：

$$\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$$

关键性质：
- **非负且和为 1**：$\mathrm{softmax}(z)_i \ge 0$，且 $\sum_i \mathrm{softmax}(z)_i = 1$。
- **平移不变性**：对任意常数 $c$，有 $\mathrm{softmax}(z) = \mathrm{softmax}(z + c)$。
  - 这也是数值稳定写法（减最大值）的理论依据。

温度参数（可选）：
$$\mathrm{softmax}_T(z)_i = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}$$
- $T$ 越小，分布越“尖”（更接近 one-hot）；$T$ 越大，分布越“平”。


In [ ]:
import numpy as np


def softmax(x, axis=-1):
    """数值稳定的 softmax。

    - 支持 1D / 2D / ND
    - 通过 `x - max(x)` 避免 exp 溢出
    """
    x = np.asarray(x, dtype=np.float64)
    x_max = np.max(x, axis=axis, keepdims=True)
    shifted = x - x_max
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)


def log_softmax(x, axis=-1):
    """数值稳定的 log_softmax，常用于交叉熵/负对数似然。"""
    x = np.asarray(x, dtype=np.float64)
    x_max = np.max(x, axis=axis, keepdims=True)
    shifted = x - x_max
    logsumexp = np.log(np.sum(np.exp(shifted), axis=axis, keepdims=True))
    return shifted - logsumexp


def softmax_naive(x, axis=-1):
    """不做稳定化的版本（用于对比，会溢出）。"""
    x = np.asarray(x, dtype=np.float64)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


In [ ]:
x = np.array(
    [
        [1, 2, 4],
        [0, 0, 0],
        [10, 1, -5],
    ]
)

p = softmax(x, axis=1)
print(p)
print('每行求和:', p.sum(axis=1))
print('每行 argmax:', p.argmax(axis=1))

In [ ]:
z = np.array([1.0, 2.0, 4.0])
print('softmax(z):      ', softmax(z))
print('softmax(z + 10): ', softmax(z + 10))
print('allclose:', np.allclose(softmax(z), softmax(z + 10)))

## 2. 数值稳定：为什么要减去最大值？

直接算 $e^{z_i}$ 会遇到**指数溢出**（`np.exp(1000)` 直接变成 `inf`）。

由于 softmax 对平移不变：
$$\mathrm{softmax}(z) = \mathrm{softmax}(z - \max(z))$$
我们可以先把最大的 logit 平移到 0，再做 `exp`，极大地降低溢出风险。

同理，很多损失函数会直接用 `log_softmax`，避免先算概率再取对数的精度问题。


In [ ]:
big = np.array([1000.0, 1001.0, 1002.0])

with np.errstate(over='ignore', invalid='ignore'):
    print('naive:', softmax_naive(big))

print('stable:', softmax(big))
print('log_softmax:', log_softmax(big))

## 3. 梯度（Jacobian）

设 $s = \mathrm{softmax}(z)$，则对任意 $i, j$：
$$\frac{\partial s_i}{\partial z_j} = s_i(\delta_{ij} - s_j)$$

矩阵形式（Jacobian）：
$$J = \mathrm{diag}(s) - s\,s^\top$$

下面用数值差分验证一下这个公式。


In [ ]:
def jacobian_softmax_numerical(z, eps=1e-6):
    z = np.asarray(z, dtype=np.float64)
    n = z.size
    J = np.zeros((n, n), dtype=np.float64)
    for j in range(n):
        zp = z.copy()
        zm = z.copy()
        zp[j] += eps
        zm[j] -= eps
        J[:, j] = (softmax(zp) - softmax(zm)) / (2 * eps)
    return J


z = np.array([0.2, -1.3, 2.5], dtype=np.float64)
s = softmax(z)
J_formula = np.diag(s) - np.outer(s, s)
J_num = jacobian_softmax_numerical(z)

print('max |J_formula - J_num| =', np.max(np.abs(J_formula - J_num)))
print('J_formula:\n', J_formula)
print('J_num:\n', J_num)

## 4. 和交叉熵（Cross-Entropy）的关系

多分类里常用 loss：
- 先对 logits 做 softmax 得到概率 $p$
- 对正确类别 $y$ 取负对数：$\mathrm{NLL} = -\log p_y$

实际实现中，通常直接用 `log_softmax`：
$$-\log \mathrm{softmax}(z)_y = -\mathrm{log\_softmax}(z)_y$$
这样更稳定、更高效。


In [ ]:
logits = np.array(
    [
        [2.0, 1.0, 0.1],
        [0.1, 0.2, 3.0],
    ],
    dtype=np.float64,
)
y = np.array([0, 2])  # 每个样本的正确类别

log_p = log_softmax(logits, axis=1)
nll = -log_p[np.arange(len(y)), y]

print('softmax 概率:\n', softmax(logits, axis=1))
print('每个样本 NLL:', nll)
print('mean NLL:', nll.mean())